In [2]:
"""
统一训练脚本：只需修改 MODEL_NAME 即可切换模型
所有其他设置（数据、优化器、调度器、增强）完全相同
数据源：generate_sionna_data.py 生成的 .npz 文件
"""
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import mltools


In [3]:
# 选择模型
MODEL_NAME = 'tcn'  # 选项: 'transformer', 'tcn', 'mamba'

# 根据 MODEL_NAME 自动导入对应模型
if MODEL_NAME == 'transformer':
    from model_transformer import build_model
elif MODEL_NAME == 'tcn':
    from model_tcn import build_model
elif MODEL_NAME == 'mamba':
    from model_mamba import build_model
else:
    raise ValueError(f"Unknown model: {MODEL_NAME}. Choose from 'transformer','tcn','mamba'")

# 环境设置
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
print(f"Using device:{device}")
print(f"Selected model:{MODEL_NAME}")

Using device:cuda
Selected model:tcn


In [ ]:
# 数据预处理
def power_normalize(signal_data):
    """功率归一化"""
    I, Q = signal_data[0], signal_data[1]
    power = np.mean(I ** 2 + Q ** 2)
    if power < 1e-10:
        return signal_data.astype(np.float32)
    return (signal_data / np.sqrt(power)).astype(np.float32)

# Python 类（Class）知识点详解

---

## 一、类的核心知识点与构建思路

### 1. 什么是类？

```python
# 类 = 数据 + 操作数据的方法 的封装体
# 类是"蓝图"，实例是"根据蓝图造出的产品"

class Dog:
    def __init__(self, name, age):   # 数据（属性）
        self.name = name
        self.age = age
    
    def bark(self):                   # 操作（方法）
        print(f"{self.name} says: Woof!")
```

### 2. 构建类的思路（以你的代码为例）

```
第一步：明确目的 → 这个类要做什么？
    → IQDataset: 管理IQ数据，提供给PyTorch DataLoader逐条读取

第二步：确定属性 → 这个类需要存储什么数据？
    → raw_data, labels, normalize, augment

第三步：确定方法 → 这个类需要什么操作？
    → __len__()     : 返回数据集长度
    → __getitem__() : 按索引取数据
    → _augment()    : 数据增强（内部辅助方法）

第四步：确定继承 → 是否复用已有的类？
    → 继承 Dataset，复用PyTorch数据加载框架
```

### 3. 类的完整结构拆解

```python
#          继承父类
#             ↓
class IQDataset(Dataset):
    
    # ============ 构造函数：初始化属性 ============
    def __init__(self, iq_data, labels, normalize=True, augment=False):
        self.raw_data = iq_data          # 实例属性：每个对象独有
        self.labels = torch.from_numpy(labels).long()
        self.normalize = normalize
        self.augment = augment
    
    # ============ 魔术方法：定义特殊行为 ============
    def __len__(self):
        return len(self.labels)           # 让 len(dataset) 生效
    
    def __getitem__(self, idx):
        ...                               # 让 dataset[0] 生效
    
    # ============ 内部方法：辅助功能 ============
    def _augment(self, sample):
        ...                               # 仅供类内部调用
```

### 4. `self` 是什么？

```python
class IQDataset(Dataset):
    def __init__(self, iq_data, labels):
        self.raw_data = iq_data   # self = "未来创建出来的那个实例对象自己"
        self.labels = labels

# 创建实例时：
dataset = IQDataset(my_data, my_labels)
# Python 自动执行：
# IQDataset.__init__(dataset, my_data, my_labels)
#                      ↑
#                  self = dataset（这个具体的实例）
```

**图示理解：**
```
dataset1 = IQDataset(data_A, labels_A)
dataset2 = IQDataset(data_B, labels_B)

dataset1.raw_data → data_A    # self 指向 dataset1
dataset2.raw_data → data_B    # self 指向 dataset2
```

### 5. 继承

```python
class Dataset:                        # 父类（PyTorch 提供）
    def __getitem__(self, idx):
        raise NotImplementedError     # 要求子类必须实现
    def __len__(self):
        raise NotImplementedError

class IQDataset(Dataset):             # 子类（你写的）
    def __getitem__(self, idx):       # ✅ 实现了父类的要求
        ...
    def __len__(self):                # ✅ 实现了父类的要求
        ...

# 好处：DataLoader 只认识 Dataset 接口
# 你只要继承 Dataset 并实现这两个方法，DataLoader 就能用你的类
```

---

## 二、双下划线魔术方法（Dunder Methods）完整梳理

> 格式：`__xxx__`，也叫 **魔术方法 / 特殊方法 / dunder method**
> 核心思想：**让你的对象支持Python的内置语法**

### 分类总结表

#### 🔧 生命周期类

| 方法 | 触发时机 | 示例 |
|------|----------|------|
| `__init__` | 创建实例时**初始化** | `obj = MyClass()` |
| `__new__` | 创建实例时**分配内存**（比init更早） | 单例模式 |
| `__del__` | 对象被**销毁**时 | `del obj` |

```python
class MyClass:
    def __init__(self, value):
        print(f"初始化: {value}")
        self.value = value
    
    def __del__(self):
        print("对象被销毁了")

obj = MyClass(42)   # → "初始化: 42"
del obj             # → "对象被销毁了"
```

#### 📏 容器/序列类（你的代码用到了这类）

| 方法 | 触发语法 | 作用 |
|------|----------|------|
| `__len__` | `len(obj)` | 返回长度 |
| `__getitem__` | `obj[idx]` | 按索引取值 |
| `__setitem__` | `obj[idx] = val` | 按索引赋值 |
| `__delitem__` | `del obj[idx]` | 按索引删除 |
| `__contains__` | `x in obj` | 成员判断 |
| `__iter__` | `for x in obj` | 迭代 |
| `__next__` | `next(obj)` | 取下一个元素 |

```python
class MyList:
    def __init__(self):
        self.data = [10, 20, 30]
    
    def __len__(self):            # len(obj)
        return len(self.data)
    
    def __getitem__(self, idx):   # obj[0], obj[1]
        return self.data[idx]
    
    def __contains__(self, item): # 3 in obj
        return item in self.data
    
    def __iter__(self):           # for x in obj
        return iter(self.data)

obj = MyList()
print(len(obj))       # → 3        调用 __len__
print(obj[1])         # → 20       调用 __getitem__
print(20 in obj)      # → True     调用 __contains__
for x in obj:         #            调用 __iter__
    print(x)          # → 10 20 30
```

#### 🖨️ 字符串表示类

| 方法 | 触发语法 | 区别 |
|------|----------|------|
| `__str__` | `print(obj)`, `str(obj)` | 给**用户**看的 |
| `__repr__` | 在控制台直接输入`obj` | 给**开发者**看的 |

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    
    def __str__(self):
        return f"点({self.x}, {self.y})"
    
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 4)
print(p)    # → "点(3, 4)"         调用 __str__
p           # → Point(x=3, y=4)   调用 __repr__（在Jupyter/交互式中）
```

#### ➕ 运算符重载类

| 方法 | 触发语法 | 运算 |
|------|----------|------|
| `__add__` | `a + b` | 加法 |
| `__sub__` | `a - b` | 减法 |
| `__mul__` | `a * b` | 乘法 |
| `__truediv__` | `a / b` | 除法 |
| `__eq__` | `a == b` | 等于 |
| `__lt__` | `a < b` | 小于 |
| `__gt__` | `a > b` | 大于 |

```python
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y
    
    def __add__(self, other):         # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)
    
    def __eq__(self, other):          # v1 == v2
        return self.x == other.x and self.y == other.y

v1 = Vector(1, 2)
v2 = Vector(3, 4)
v3 = v1 + v2       # → Vector(4, 6)   调用 __add__
print(v1 == v2)     # → False          调用 __eq__
```

#### 🔑 属性访问类

| 方法 | 触发语法 |
|------|----------|
| `__getattr__` | 访问**不存在**的属性时 |
| `__setattr__` | 设置任何属性时 |
| `__hasattr__` | `hasattr(obj, name)` |

#### 📦 上下文管理器类

| 方法 | 触发语法 |
|------|----------|
| `__enter__` | `with obj as x:` 进入时 |
| `__exit__` | `with` 块结束时 |

```python
class FileManager:
    def __init__(self, filename):
        self.filename = filename
    
    def __enter__(self):
        self.file = open(self.filename, 'r')
        return self.file
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.file.close()

with FileManager("data.txt") as f:   # __enter__
    content = f.read()
# 自动调用 __exit__，确保文件关闭
```

#### 🔢 其他常用

| 方法 | 触发语法 |
|------|----------|
| `__call__` | `obj()` 像函数一样调用对象 |
| `__hash__` | `hash(obj)` |
| `__bool__` | `if obj:` |

```python
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    
    def __call__(self, x):        # 让对象可以像函数一样被调用
        return x * self.factor

double = Multiplier(2)
print(double(5))    # → 10       调用 __call__
print(double(8))    # → 16       调用 __call__
```

---

## 三、单下划线 `_augment` 的含义

### Python 命名约定（Convention）

```python
class IQDataset(Dataset):
    
    def _augment(self, sample):      # 单下划线开头
        """这是一个"内部方法"，表示：
        - 这个方法仅供类内部使用
        - 外部不应该直接调用（但技术上可以调用）
        - 是一种"君子协定"，不是强制限制
        """
        ...
    
    def __getitem__(self, idx):
        ...
        sample = self._augment(sample)  # ✅ 类内部调用
        ...

# 外部：
dataset = IQDataset(...)
dataset._augment(sample)   # ⚠️ 技术上可以，但不建议
```

### 完整对比

```python
class Example:
    def public_method(self):       # 公开方法：随便调用
        pass
    
    def _private_method(self):     # 单下划线：约定私有（建议不要外部调用）
        pass                       # "我是内部实现细节，可能随时改变"
    
    def __mangled_method(self):    # 双下划线：名称改写（Name Mangling）
        pass                       # Python 会改名为 _Example__mangled_method
                                   # 防止子类意外覆盖
    
    def __dunder_method__(self):   # 双下划线前后：魔术方法
        pass                       # Python 内置协议，有特殊用途
```

**实际验证名称改写：**
```python
class Parent:
    def __secret(self):
        print("Parent's secret")

class Child(Parent):
    def __secret(self):          # 不会覆盖父类的 __secret
        print("Child's secret")

obj = Child()
obj._Child__secret()    # → "Child's secret"
obj._Parent__secret()   # → "Parent's secret"
# 两个 __secret 被改写成了不同的名字，互不干扰
```

### 为什么 `_augment` 用单下划线？

```
_augment 是数据增强的内部实现细节
  ↓
只在 __getitem__ 内部被调用
  ↓
外部用户不需要知道它的存在
  ↓
加 _ 前缀 → 告诉其他开发者："这是内部方法，别直接调"
```

---

## 四、类怎么被使用？参数怎么传？传到哪里？

### 完整使用流程

```python
# ============ 第 1 步：准备数据 ============
iq_data = np.random.randn(1000, 2, 1024)   # 1000个样本
labels  = np.random.randint(0, 10, 1000)    # 1000个标签

# ============ 第 2 步：创建实例 ============
dataset = IQDataset(iq_data, labels, normalize=True, augment=True)
#                     ↓       ↓        ↓              ↓
#                  iq_data  labels  normalize=True  augment=True
#                     ↓       ↓        ↓              ↓
#           self.raw_data  self.labels  self.normalize  self.augment
```

### 参数传递的完整流程图

```
dataset = IQDataset(iq_data, labels, normalize=True, augment=True)
                ↓
    Python 自动调用：
    IQDataset.__init__(dataset, iq_data, labels, normalize=True, augment=True)
                         ↑        ↑       ↑          ↑              ↑
                        self   iq_data  labels    normalize      augment
                         ↓
    执行函数体：
        self.raw_data  = iq_data       →  dataset.raw_data = iq_data
        self.labels    = labels转tensor →  dataset.labels   = tensor
        self.normalize = True          →  dataset.normalize = True
        self.augment   = True          →  dataset.augment   = True
```

### 使用实例的各种方式

```python
# ============ 方式1：直接索引访问 ============
sample, label = dataset[0]     # 触发 __getitem__(self, 0)
sample, label = dataset[99]    # 触发 __getitem__(self, 99)

# ============ 方式2：获取长度 ============
print(len(dataset))            # 触发 __len__(self) → 1000

# ============ 方式3：配合 DataLoader 使用（最常见）============
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,              # 传入你的 Dataset 实例
    batch_size=32,        # 每批32个样本
    shuffle=True,         # 打乱顺序
    num_workers=4         # 多进程加载
)

# DataLoader 内部自动做的事情：
# 1. 调用 len(dataset)          → 知道有1000个样本
# 2. 生成随机索引 [573, 12, 847, ...]
# 3. 逐个调用 dataset[573], dataset[12], dataset[847], ...
# 4. 把32个结果堆叠成一个 batch

for batch_data, batch_labels in loader:
    # batch_data:   shape (32, 1024, 2)  ← 32个样本堆叠
    # batch_labels: shape (32,)          ← 32个标签堆叠
    
    output = model(batch_data)
    loss = criterion(output, batch_labels)
    ...
```

### 完整的调用链路图

```
for data, labels in DataLoader(dataset, batch_size=32):
    │
    ├─→ len(dataset)                     # 调用 __len__
    │       → return 1000
    │
    ├─→ 生成32个随机索引 [idx1, idx2, ...]
    │
    ├─→ dataset[idx1]                    # 调用 __getitem__
    │       ├─→ sample = self.raw_data[idx1].copy()
    │       ├─→ if self.augment:
    │       │       sample = self._augment(sample)  # 调用内部方法
    │       │           ├─→ 随机平移
    │       │           ├─→ 随机加噪
    │       │           └─→ 随机旋转
    │       ├─→ if self.normalize:
    │       │       sample = power_normalize(sample)
    │       ├─→ tensor转换 + permute
    │       └─→ return (sample_tensor, label_tensor)
    │
    ├─→ dataset[idx2] ...（重复32次）
    │
    └─→ 将32个结果 stack 成 batch → (batch_data, batch_labels)
```

### 训练 vs 验证的不同配置

```python
# 训练集：开启增强
train_dataset = IQDataset(train_data, train_labels, 
                          normalize=True, augment=True)   # ← augment=True

# 验证集：关闭增强
val_dataset = IQDataset(val_data, val_labels, 
                        normalize=True, augment=False)    # ← augment=False

# 同一个类，不同参数 → 不同行为
# 这就是类的灵活性
```

In [ ]:
class IQDateset(Dataset):
    def __init__(self, iq_data, labels, normalize=True, augment=False):
        self.raw_data = iq_data
        self.labels = torch.from_numpy(labels).long()
        self.normalize = normalize
        self.augment = augment

    def __len__(self):
        return len(self.labels)
    
    def _augment(self, sample):
        if np.random.random() < 0.5:
            shift = np.random.randint(-50, 50)
            sample = np.roll(sample, shift, axis=1)
        if np.random.random() < 0.5:
            signal_power = np.mean(sample ** 2)
            if signal_power > 1e-10:
                snr_db = np.random.uniform(10, 30)
                noise_power = signal_power / (10 ** (snr_db / 10))
                sample = sample + np.random.normal(0, np.sqrt(noise_power), sample.shape)
        if np.random.random() < 0.5:
            theta = np.random.uniform(0, 2 * np.pi)
            I, Q = sample[0].copy(), sample[1].copy()
            sample[0] = I * np.cos(theta) - Q * np.sin(theta)
            sample[1] = I * np.sin(theta) - Q * np.cos(theta)
        return sample
    
    def __getitem__(self, idx):
        sample = self.raw_data[idx].copy()
        if self.augment:
            sample = self._augment(sample)
        if self.normalize:
            sample = power_normalize(sample)
        sample_tensor = torch.from_numpy(sample).float()
        sample_tensor = sample_tensor.permute(1, 0) # (2,1024) → (1024,2)
        return sample_tensor, self.labels[idx]

# `_augment` 信号增强详解

---

## 一、三种增强的独立性

**是的，每个 `np.random.random()` 都是独立调用，生成独立的随机值。**

```python
if np.random.random() < 0.5:    # 随机值 ①，比如 0.32 → 执行
    ...  # 增强1：时间平移

if np.random.random() < 0.5:    # 随机值 ②，比如 0.71 → 跳过
    ...  # 增强2：加噪声

if np.random.random() < 0.5:    # 随机值 ③，比如 0.15 → 执行
    ...  # 增强3：相位旋转
```

三个 `if` 互不影响，因此存在 **$2^3 = 8$ 种组合**：

| 平移 | 加噪 | 旋转 | 概率 |
|:---:|:---:|:---:|:---:|
| ✗ | ✗ | ✗ | 12.5% |
| ✓ | ✗ | ✗ | 12.5% |
| ✗ | ✓ | ✗ | 12.5% |
| ✗ | ✗ | ✓ | 12.5% |
| ✓ | ✓ | ✗ | 12.5% |
| ... | ... | ... | ... |
| ✓ | ✓ | ✓ | 12.5% |

---

## 二、增强1：循环时间平移

```python
shift = np.random.randint(-50, 50)
sample = np.roll(sample, shift, axis=1)
```

### 物理逻辑

> 模拟**信号采集起始时刻不确定**——同一个信号，接收机可能从不同时刻开始截取

```
sample shape = (2, 1024)
                ↑    ↑
               I/Q  时间采样点

axis=1 就是沿时间轴平移
```

### 数学表示

$$x'[n] = x[(n - \Delta) \mod N], \quad \Delta \in [-50, 49]$$

### 图示

```
原始信号:     [A B C D E F G H]
shift = +3:   [F G H | A B C D E]    ← 右移3，尾部绕到头部
shift = -2:   [C D E F G H | A B]    ← 左移2，头部绕到尾部
                              ↑
                          循环(circular)
```

### 为什么用循环移位而非截断？

```
截断移位:  [0 0 0 A B C D E]    ← 丢失信息，引入零值
循环移位:  [F G H A B C D E]    ← 信息完整保留

对于周期性/连续采集的信号，循环移位更合理
```

---

## 三、增强2：加性高斯白噪声（AWGN）

```python
signal_power = np.mean(sample ** 2)
if signal_power > 1e-10:
    snr_db = np.random.uniform(10, 30)
    noise_power = signal_power / (10 ** (snr_db / 10))
    sample = sample + np.random.normal(0, np.sqrt(noise_power), sample.shape)
```

### 物理逻辑

> 模拟**无线信道中的热噪声、电子噪声**——真实接收信号总是混有噪声

$$y(t) = x(t) + n(t)$$

### 逐行数学推导

**第1步：计算信号功率**

$$P_s = \frac{1}{N}\sum_{i} x_i^2 = \text{np.mean(sample}^2\text{)}$$

```
sample = [[I₁, I₂, ..., I₁₀₂₄],      ← I路
           [Q₁, Q₂, ..., Q₁₀₂₄]]      ← Q路

P_s = 所有 2×1024 = 2048 个值的平方的均值
```

**第2步：随机选择信噪比**

$$\text{SNR}_{\text{dB}} \sim \text{Uniform}(10, 30)$$

> 10dB = 噪声较大（信号是噪声的10倍）
> 30dB = 噪声很小（信号是噪声的1000倍）

**第3步：从 SNR 反推噪声功率**

SNR 的定义：

$$\text{SNR}_{\text{dB}} = 10 \cdot \log_{10}\left(\frac{P_s}{P_n}\right)$$

反推：

$$\frac{\text{SNR}_{\text{dB}}}{10} = \log_{10}\left(\frac{P_s}{P_n}\right)$$

$$10^{\text{SNR}_{\text{dB}}/10} = \frac{P_s}{P_n}$$

$$\boxed{P_n = \frac{P_s}{10^{\text{SNR}_{\text{dB}}/10}}}$$

```python
noise_power = signal_power / (10 ** (snr_db / 10))
```

**第4步：生成噪声并叠加**

$$n \sim \mathcal{N}(0, \,\sigma), \quad \sigma = \sqrt{P_n}$$

```python
noise = np.random.normal(0, np.sqrt(noise_power), sample.shape)
sample = sample + noise
```

### 数值示例

```
假设 signal_power = 1.0
随机得到 snr_db = 20

noise_power = 1.0 / (10^(20/10))
            = 1.0 / (10^2)
            = 1.0 / 100
            = 0.01

σ = √0.01 = 0.1

噪声 ~ N(0, 0.1)    ← 标准差为信号幅度的约10%
```

### 为什么要检查 `signal_power > 1e-10`？

```python
if signal_power > 1e-10:   # 防止全零信号
```
> 如果信号功率接近零，计算 SNR 无意义，`noise_power` 也接近零，加噪无效果。避免数值问题。

---

## 四、增强3：IQ 相位旋转

```python
theta = np.random.uniform(0, 2 * np.pi)
I, Q = sample[0].copy(), sample[1].copy()
sample[0] = I * np.cos(theta) - Q * np.sin(theta)
sample[1] = I * np.sin(theta) - Q * np.cos(theta)   # ⚠️ 注意这里
```

### 物理逻辑

> 模拟**载波相位偏移**——发送端和接收端的本振频率不完全同步，导致IQ平面上的整体旋转

### 标准数学公式

IQ信号本质是**复信号**：

$$z = I + jQ$$

载波相位偏移 $\theta$ 等价于复数乘法：

$$z' = z \cdot e^{j\theta} = (I + jQ)(\cos\theta + j\sin\theta)$$

展开：

$$\begin{cases} I' = I\cos\theta - Q\sin\theta \\ Q' = I\sin\theta + Q\cos\theta \end{cases}$$

矩阵形式（**标准旋转矩阵**）：

$$\begin{pmatrix} I' \\ Q' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} I \\ Q \end{pmatrix}$$

### ⚠️ 代码中可能有 Bug

```python
# 你的代码：
sample[1] = I * np.sin(theta) - Q * np.cos(theta)    # ← 减号

# 标准旋转应该是：
sample[1] = I * np.sin(theta) + Q * np.cos(theta)    # ← 加号
```

对比：

| | I' | Q' |
|---|---|---|
| **标准旋转** | $I\cos\theta - Q\sin\theta$ | $I\sin\theta \mathbf{+} Q\cos\theta$ |
| **你的代码** | $I\cos\theta - Q\sin\theta$ | $I\sin\theta \mathbf{-} Q\cos\theta$ |

你的代码对应的矩阵：

$$\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & -\cos\theta \end{pmatrix}$$

行列式 $= -\cos^2\theta - (-\sin^2\theta) = \sin^2\theta - \cos^2\theta = -\cos 2\theta \neq 1$

**这不是纯旋转，而是旋转+反射**，会改变信号的结构。作为数据增强凑巧可能仍有效，但物理上不正确。

### IQ 平面的几何直觉

```
        Q
        ↑        旋转 θ
        |      ↗
        |    /  z' = z·e^(jθ)
        |  / θ
        |/______ → I
        z(I,Q)

整个IQ星座图绕原点旋转 θ 角
所有采样点同步旋转，信号"形状"不变
```

### 为什么要 `.copy()`？

```python
I, Q = sample[0].copy(), sample[1].copy()
```

```
如果不 copy：
  I = sample[0]           ← I 是 sample[0] 的引用（指向同一块内存）
  sample[0] = I*cos - Q*sin   ← 修改了 sample[0]，也就修改了 I ！
  sample[1] = I*sin + Q*cos   ← 此时 I 已经不是原始值了 ❌

使用 copy：
  I = sample[0].copy()    ← I 是独立副本
  sample[0] = ...         ← 修改 sample[0] 不影响 I ✅
  sample[1] = ...         ← I 仍是原始值 ✅
```

---

## 五、三种增强的总结对比

```
┌─────────────┬──────────────────┬──────────────────────────┐
│   增强方法   │   模拟的物理现象  │      数学操作            │
├─────────────┼──────────────────┼──────────────────────────┤
│  循环平移    │ 采集起始时刻偏移  │ x'[n] = x[(n-Δ) mod N]  │
│             │                  │ Δ ∈ [-50, 49]            │
├─────────────┼──────────────────┼──────────────────────────┤
│  加高斯噪声  │ 信道热噪声       │ y = x + n               │
│             │                  │ n ~ N(0, √Pn)           │
│             │                  │ Pn = Ps / 10^(SNR/10)   │
├─────────────┼──────────────────┼──────────────────────────┤
│  IQ相位旋转  │ 载波相位不同步    │ z' = z · e^(jθ)         │
│             │                  │ θ ~ Uniform(0, 2π)      │
└─────────────┴──────────────────┴──────────────────────────┘
```

In [ ]:
print("loading data")